# 第 06 步：同一日期的 40 股动量排名

<!-- SNAPSHOT_NOTE -->

## 数据准备

本页记录的是完整快照上的实际运行。从 GitHub 克隆源码后，先在仓库根目录运行 `python publication/restore_snapshot.py`，恢复同版本的 `data/` 与 `runs/`，再执行本步。[完整数据与大型结果索引](../../publication/ASSET_INDEX.md)保留全部路径和哈希；本页图表与 Notebook 已保存输出可直接阅读。

## 目标与两种比较方向

上一节只观察 `SYN001` 随时间变化的动量。本步将同一套公式逐只应用于 40 只人工股票，再选定同一个日期，按动量从高到低排序。问题由“这只股票最近涨跌多少”进一步变成“这只股票相对其他股票处在什么位置”。

这里有两种比较方向：沿同一股票的日期观察，是时间序列比较；固定同一日期、比较多只股票，是横截面比较。不同日期算出的动量不能随意放在同一个排名中，因为它们对应不同的市场时点和信息集合。

输入仍是固定人工样本，当前选定 `2020-01-31`。这个日期属于模拟工作日日历，不代表真实 A 股当天开市，也不是实际交易建议。脚本只展示前十名，尚未分配资金、发出订单或扣除成本。

完成 [第 05 步](../05_momentum_factor/README.md) 后，在仓库根目录执行：

```powershell
.\.venv\Scripts\python.exe examples/06_cross_section_ranking.py
```

## 因子与排名规则

每只股票 i 使用同一个历史公式：

$$
M_{i,t}^{(20)}=\frac{P_{i,t}}{P_{i,t-20}}-1.
$$

相应 21 个日历价格点仍需全部有效。当前动量方向暂定为“较高值排前”，这是一个需要未来检验的研究假设。排名本身只是一种历史特征的整理，不证明排在前面的股票随后收益更高。

展示顺序是：先按动量降序，再按代码升序。第二个条件只是解决并列分数时的确定顺序，不表示代码更小的股票具有更强金融信号。本节的 `rank=1,2,...` 是展示序号；下一节计算 Spearman 秩相关时，真正的并列值需要平均秩，不能直接沿用代码打破并列后的展示序号。

## 先按股票分组

```python
factor_tables = []
for symbol, stock in prices.groupby("symbol", sort=True):
    stock = stock.sort_values("date").set_index("date")
    stock = stock.reindex(calendar["date"]).reset_index()
    stock["symbol"] = symbol
```

先创建列表，收集每只股票的小表。`groupby("symbol")` 保证每次计算只面对一个代码，`sort=True` 让分组按代码有序出现。它不是“按代码挑出表现好的一组”，而是防止同一滚动公式跨越股票边界。

循环里先按日期排序，再设日期索引，随后对齐同一份独立日历。重新赋值 `stock["symbol"]=symbol`，是在对齐补出缺失行后明确它属于哪只股票；这不填补价格，也不把未知行情变成已知。特征计算仍只使用当前行及此前的历史数据。

如果在 73,080 行混合表上直接做 `shift(20)`，很容易把另一只股票的价格当成本股历史价格。即使算出的数字看起来平滑，也没有正确的金融含义。分组、时间排序和独立日历是公式成立的前提，不能靠最终收益好看来替代。

## 逐股复用完整窗口规则

```python
close = stock["adj_close"].where(stock["adj_close"] > 0)
stock["start_date"] = stock["date"].shift(lookback)
stock["price_20_sessions_ago"] = close.shift(lookback)
valid_price_count = close.rolling(lookback + 1, min_periods=lookback + 1).count()
stock["momentum_20"] = (close / stock["price_20_sessions_ago"] - 1).where(
    valid_price_count == lookback + 1
)
```

先排除非正价格。`start_date` 和历史价格使用相同的位移，所以后续能把一个排名值追溯到具体起止日期。`valid_price_count` 要求连续 21 个有效价格位置，避免中间缺口被端点比值掩盖。最后保存的动量仍是小数，没有在计算过程中混入百分数。

```python
factor_values = pd.concat(factor_tables, ignore_index=True)
signal_date = pd.Timestamp("2020-01-31")
snapshot = factor_values.loc[factor_values["date"] == signal_date].copy()
ranked = snapshot.dropna(subset=["momentum_20"]).copy()
```

`concat` 按行拼接各股因子表，`ignore_index=True` 重新生成统一行标签。整张表含多个历史日期没有问题，关键是每个因子值都只由当时及此前的数据生成。随后选定日期，只比较该截面，并排除无法计算动量的记录。整个过程不使用未来收益筛选当前名单，也不把缺失因子补为零后参加排名。

## 建立确定的展示顺序

```python
ranked = ranked.sort_values(
    ["momentum_20", "symbol"], ascending=[False, True]
).reset_index(drop=True)
ranked["rank"] = range(1, len(ranked) + 1)
ranked["momentum_20_pct"] = ranked["momentum_20"] * 100
```

`False` 对应动量降序，`True` 对应代码升序，两个布尔值分别作用于两个排序字段。重排行标签后，用 `range` 生成从 1 开始的展示名次。百分数列只用于打印，不参与排序计算。只打印 `head(10)` 时，它不会把内存里的其他 30 只股票删除；截面与有效样本数仍应单独记录。

## 实际截面结果

当前 `2020-01-31` 截面有 **40 条记录、40 个有效动量、0 个缺失动量**。前十名如下，百分数为实际小数因子乘 100 后的适度舍入展示：

| 展示名次 | 人工代码 | 20 日动量 |
| --- | --- | ---: |
| 1 | SYN037 | 15.703985% |
| 2 | SYN008 | 10.858528% |
| 3 | SYN034 | 8.161741% |
| 4 | SYN025 | 1.816338% |
| 5 | SYN029 | 1.328662% |
| 6 | SYN001 | -0.354572% |
| 7 | SYN018 | -0.420423% |
| 8 | SYN004 | -0.770839% |
| 9 | SYN011 | -1.953194% |
| 10 | SYN015 | -2.787643% |

我观察到前十名中有负动量，这并不矛盾：排名表达的是相对位置，不要求绝对上涨。`SYN001` 过去 20 个模拟交易区间略有下跌，但仍优于当前样本中多数股票。反过来，某只股票上涨也不意味着它在当期横截面里排前。

这张表不是策略收益表。第一名的 15.703985% 描述已经发生的历史窗口，不是之后一个月会赚到的数字。不能将其直接当作预期收益，更不能把前十名历史涨幅平均后称为选股回测收益。

## 从 SYN001 排名追溯原始价格

| 追溯字段 | 实际记录 |
| --- | ---: |
| 当前信号日期 | 2020-01-31 |
| 窗口起点 | 2020-01-03 |
| 起点复权收盘价 | 21.59631562275345 |
| 终点复权收盘价 | 21.51974115509771 |
| 20 日动量小数 | -0.00354571904733 |
| 动量百分数 | -0.3545719047% |
| 当日展示名次 | 6 / 40 |

用价格比再次复算 `21.51974115509771/21.59631562275345-1`，得到与表中一致的动量。这个检查把“排名第六”重新连接到一只具体股票、一段日期和两个价格，避免排序以后只剩名次而失去数据来源。

全历史因子表有 **73,080 条记录**，其中 **800 条动量缺失**，有效 **72,280 条**。当前人工数据中，每只股票最初 20 行都在预热，`40×20=800`，因此这个缺失总量与规则一致。若换成真实数据，不能照搬“全部缺失都只是预热”的结论，还要核对每股覆盖、停牌、价格和成员资格。

## 股票池与相对名次的限制

当前四十只人工股票从生成开始就属于固定样本，因此本节没有解决真实历史成员资格问题。真实研究中，需要确认某只股票在信号时点是否存在、是否属于研究范围，以及退市或被剔除的股票是否仍保留在历史样本里。某只股票后来的数据齐全、今天仍能找到，并不足以倒推它在所有过去日期都应该进入同一股票池。

名次也依赖当期比较对象。第六名在四十只股票中的位置，不能直接与第六名在三百只股票中的位置视为同一强弱。即使原始动量完全没变，股票池扩大或缩小也会改变相对顺序。因此要保存截面总数、有效数和缺失数，不能只复制前十行而不交代母样本。

名次之间的差也不能当作相等的经济差距。第一名和第二名可能相差很大，第六名和第七名可能仅差一点；展示序号并没有保留这些距离信息。后续选择标准化分数、秩或原始值时，需要说明各自表达的信息。本节同时保存动量小数与排名，正是为了让相对位置仍能追溯到实际因子值。

## 常见错误与修正

| 错误操作 | 为什么会错 | 修正方式 |
| --- | --- | --- |
| 把所有日期的动量一起排前十 | 混合了不同信息时点 | 先固定一个截面日期 |
| 对全表直接位移 | 历史价格可能来自别的股票 | 先按代码分组再计算 |
| 删除缺价日期后再数 20 行 | 20 行不再等于约定日历中的 20 个区间 | 先对齐独立日历 |
| 把缺失动量补 0 | 未知值会获得一个虚假的中间位置 | 排除无效值并报告数量 |
| 把展示名次用于并列秩相关 | 代码顺序被错误解释为信号强弱 | 统计相关时按分数重新计算平均秩 |
| 把 Top 10 当成已经买入 | 当前脚本没有资金、订单或成本 | 将排名与实际成交分开 |

## 验收记录与下一步

本步验收包括：解释时间序列与横截面的区别，将每个排名追溯到该股历史窗口，核对有效样本数和预热缺失，并说明负动量为何仍可能排在前列。当前结果是一份同一信息时点下、可以复现的历史特征排序。

下一步进入 [第 07 步脚本：未来收益与 Rank IC](../../examples/07_forward_returns_rank_ic.py)。接下来建立从信号之后开始的未来标签，再检验当前排名与未来收益排名的关系；完成这一步之前，历史排名还不能被解释为预测成功。

## 完整可运行实现

上面的片段用于解释思路；下面各单元按顺序执行完整脚本。先定位仓库根目录，并显式设置脚本路径，使 `__file__` 在 Notebook 中也有明确含义。

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "pyproject.toml").exists()
                     and (candidate / "examples/01_read_prices.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("我需要从本项目目录或它的子目录打开 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
__file__ = str(PROJECT_ROOT / 'examples/06_cross_section_ranking.py')
sys.argv = [__file__]


### 完整实现 · 单元 1

In [2]:
"""第六课：逐股票计算动量，再在同一天做横截面排名。"""

from pathlib import Path
import pandas as pd

project_dir = Path(__file__).resolve().parents[1]
data_path = project_dir / "runs" / "first_demo_20261107" / "input" / "prices.csv"
prices = pd.read_csv(data_path, dtype={"symbol": "string"}, parse_dates=["date"])
calendar = pd.read_csv(data_path.with_name("calendar.csv"), parse_dates=["date"])
lookback = 20

# 1. 每次取出一只股票的历史，复用上一课的计算方法。
factor_tables = []
for symbol, stock in prices.groupby("symbol", sort=True):
    stock = stock.sort_values("date").set_index("date")
    stock = stock.reindex(calendar["date"]).reset_index()
    stock["symbol"] = symbol  # 标明所属股票；不填补任何缺失价格。
    close = stock["adj_close"].where(stock["adj_close"] > 0)
    stock["start_date"] = stock["date"].shift(lookback)
    stock["price_20_sessions_ago"] = close.shift(lookback)
    valid_price_count = close.rolling(lookback + 1, min_periods=lookback + 1).count()
    stock["momentum_20"] = (close / stock["price_20_sessions_ago"] - 1).where(
        valid_price_count == lookback + 1
    )
    factor_tables.append(stock[[
        "date", "symbol", "start_date", "adj_close", "price_20_sessions_ago", "momentum_20",
    ]])

# 2. 将各股票的小表按行拼接，得到全部日期、全部股票的因子表。
factor_values = pd.concat(factor_tables, ignore_index=True)

# 3. 只选同一天；本日期属于模拟工作日日历，不是实际交易安排。
signal_date = pd.Timestamp("2020-01-31")
snapshot = factor_values.loc[factor_values["date"] == signal_date].copy()
ranked = snapshot.dropna(subset=["momentum_20"]).copy()

### 完整实现 · 单元 2

In [3]:
if ranked.empty:
    raise ValueError("该日没有有效因子，请检查日历、预热长度和缺失行情。")

### 完整实现 · 单元 3

In [4]:
# 4. 动量从高到低；同分时按代码排序，保证展示顺序确定。
# rank 是展示的顺序号；同分后按代码排序不代表信号强度不同。
ranked = ranked.sort_values(
    ["momentum_20", "symbol"], ascending=[False, True]
).reset_index(drop=True)
ranked["rank"] = range(1, len(ranked) + 1)
ranked["momentum_20_pct"] = ranked["momentum_20"] * 100

# 5. 前十行仅用于展示排序，未在本脚本执行任何组合交易。
columns = ["rank", "symbol", "momentum_20_pct"]
print("数据类型：40 只人工股票；动量百分数列的单位为 %。")
print("横截面日期：", signal_date.date())
print("当日记录数：", len(snapshot))
print("当日有效因子数：", len(ranked))
print("当日缺失因子数：", int(snapshot["momentum_20"].isna().sum()))
print("\n当日动量前 10 名：")
print(ranked[columns].head(10).to_string(index=False))

# 6. 回到熟悉的 SYN001，解释它在同日股票池中的相对位置。
focus_symbol = "SYN001"
focus = ranked.loc[ranked["symbol"] == focus_symbol].iloc[0]
print(f"\n{focus_symbol}：第 {focus['rank']} 名，20 日动量 {focus['momentum_20']:.6%}")
print(f"计算区间：{focus['start_date']:%Y-%m-%d} 至 {focus['date']:%Y-%m-%d}")
print(f"起点价格 {focus['price_20_sessions_ago']:.12f}，终点价格 {focus['adj_close']:.12f}")
print("\n全部历史因子记录数：", len(factor_values))
print("全部历史缺失因子数：", int(factor_values["momentum_20"].isna().sum()))

数据类型：40 只人工股票；动量百分数列的单位为 %。
横截面日期： 2020-01-31
当日记录数： 40
当日有效因子数： 40
当日缺失因子数： 0

当日动量前 10 名：
 rank symbol  momentum_20_pct
    1 SYN037        15.703985
    2 SYN008        10.858528
    3 SYN034         8.161741
    4 SYN025         1.816338
    5 SYN029         1.328662
    6 SYN001        -0.354572
    7 SYN018        -0.420423
    8 SYN004        -0.770839
    9 SYN011        -1.953194
   10 SYN015        -2.787643

SYN001：第 6 名，20 日动量 -0.354572%
计算区间：2020-01-03 至 2020-01-31
起点价格 21.596315622753，终点价格 21.519741155098

全部历史因子记录数： 73080
全部历史缺失因子数： 800


## 运行复核

以上实现已在干净内核中按顺序执行；执行状态记录在仓库的 [Notebook 校验记录](../../publication/notebook_validation.json)。代码中的数据身份、费用和成交假设仍然适用，运行成功只验证对应实现。